# Tarea 2 · Parte 2 — Evaluación y extracción estructurada

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook mide qué tan bien leen los modelos (CER en documentos, WER en audio), los compara sobre **las mismas muestras**, extrae campos estructurados con un LLM y audita esa extracción a mano.

| | |
|---|---|
| **Fecha** | 27 de octubre de 2026 |
| **Datos** | `datos/documentos.csv` (25 documentos) y `datos/transcripciones.csv` (20 clips, 32.3 min) de la Parte 1 |
| **Referencias** | Transcripción manual propia de 10 documentos y 10 clips (`datos/raw/referencias/`, no versionada: puede tener nombres reales) |

## 1. Muestra de evaluación y configuraciones (fijadas antes de ver métricas)

Esta sección se escribió y se subió a GitHub **antes** de terminar las transcripciones de referencia y antes de calcular cualquier CER o WER. Ninguna configuración se cambia después según los resultados.

### Muestra

- **Documentos (10):** 3 impresos, 3 fotos y 4 manuscritos. Los manuscritos tienen un elemento más porque son el caso difícil.
- **Audios (10):** 7 fuentes o autores distintos y 5 condiciones: calle, fiesta con ruido, marcha con ruido, testimonio y lectura en estudio. Incluye clips con 2 o más hablantes.
- **Criterio:** dentro de cada tipo se eligieron los elementos de **menor costo de transcripción manual**. Así quedaron fuera `doc_003`, una página casi vacía, y las fotos de más de 5 000 caracteres (`doc_021`–`doc_025`). En audio se excluyeron la conferencia (`clip_003`) y los testimonios de 3–4 min (`clip_014`, `clip_015`), y por la misma razón se tomaron 2 de las 6 lecturas de OpenSLR.
- **Sesgo conocido:** el criterio favorece páginas y audios cortos. Las métricas describen esta muestra, no necesariamente a las páginas de periódico más densas.
- **Recortes (cambio antes de medir, 27/10/2026):** transcribir a mano las 10 páginas completas (unas 2 400 palabras) no fue viable. Por eso se evalúa un **fragmento fijo de cada uno de los 10 documentos**: la parte superior de la página, con unas 70 palabras (`recortar_muestra.py`). Los documentos de menos de 90 palabras (`doc_001`, `doc_020`) van completos. El corte cae siempre entre dos líneas. En 6 páginas se calculó con la densidad de texto, y en 4 se eligió a mano mirando solo la imagen, porque los títulos grandes ocupan mucha altura con pocas palabras. Los 3 OCR se vuelven a ejecutar **sobre los mismos recortes** y la referencia transcribe exactamente el recorte. Se mantienen los 10 documentos y los 3 tipos. A cambio, el fragmento superior tiene más títulos y encabezados que una página promedio.

### Transcripción de referencia
Se hace a mano y sin mirar la salida de ningún modelo. Se copia línea por línea y con la ortografía original. Las palabras ilegibles se marcan `[?]` y los tramos que no se entienden, `[inaudible]`. En audio se omiten las muletillas (`eh`, `mmm`) y los números se escriben con cifras. La guía completa está en `datos/raw/referencias/LEEME.md`.

### Normalización (la misma para referencia e hipótesis)
Se usa `metricas.normalizar`:
- quita las marcas `[HABLANTE_n]`, `[?]` y `[inaudible]`;
- convierte los números a palabras (`19` → `diecinueve`);
- pasa todo a minúsculas, quita la puntuación y deja espacios simples;
- **quita las tildes y diacríticos** (`á` → `a`, `ñ` → `n`).

CER y WER se calculan **sobre los textos originales**, sin seudonimizar, para que el reemplazo de nombres no cuente como error. El valor global agrega los errores de los 10 casos (jiwer) y también se reporta el valor de cada caso.

> **Cambio antes de medir (27/10/2026):** el plan inicial conservaba las tildes y no convertía números. Al revisar las primeras referencias manuales se vio que las tildes no estaban marcadas de forma pareja y que algunos números estaban en palabras y otros en cifras. Las dos cosas son diferencias de tecleo, no errores de lectura del modelo, así que la normalización las iguala. El cambio se hizo **antes de calcular cualquier CER o WER**.

### Modelos

| Modalidad | Rol | Configuración |
|---|---|---|
| OCR | **Línea base** | PaddleOCR 3.0.3 por defecto: PP-OCRv5 *mobile* det + rec, orientación de página, desdoblado UVDoc, orientación de línea y orden de lectura de Paddle. Solo se cambian los modelos *server* por los *mobile*, porque los *server* no corren en este CPU (Parte 1). |
| OCR | Ajustado | PaddleOCR de la Parte 1: *mobile*, detección a 1600 px, sin preprocesado de página y orden de lectura por columnas. Se definió en la Parte 1, antes de que existieran las referencias. |
| OCR | Alternativo | Tesseract 5.5.1 (tesserocr 2.9.1, `spa` de tessdata 4.1.0, LSTM, PSM 3) |
| ASR | **Línea base** | faster-whisper `small` (int8, CPU, `language=es`, VAD Silero, beam 5) |
| ASR | Mejorado | faster-whisper `large-v3-turbo`, con la misma configuración (el modelo de la Parte 1) |
| ASR | Alternativo | Qwen3-ASR 0.6B (transformers, float32, CPU, `language=es`, tramos VAD ≤ 12 s, decodificación greedy) |

La latencia de ASR se mide solo durante la transcripción, sin contar la carga del modelo ni la diarización. Los modelos se ejecutan uno por uno en el mismo equipo (Intel i5 de 2 núcleos). **Factor de tiempo real = segundos de proceso / duración del audio.**

### Extracción con LLM
- **Modelo:** Gemini 3.8 Flash (`gemini-3.8-flash`, API gratuita), temperatura 0 y salida JSON con esquema fijo.
- **Entrada:** los textos **ya seudonimizados**, para no enviar nombres reales a un servicio externo.
- **Campos comunes:** `modalidad`, `tema` y `resumen`.
- **Campos de documentos:** `emisor`, `fecha`, `lugar` y `monto`.
- **Campos de audio:** `numero_hablantes` y `citas_clave`.
- Un campo que no aparece en el texto debe quedar en `null`.

La auditoría humana revisa 20 extracciones sorteadas con semilla fija (10 documentos y 10 audios).

In [1]:
import pandas as pd

docs = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
fuentes = pd.read_csv("datos/fuentes.csv").set_index("id")

MUESTRA_DOCS = ["doc_001", "doc_002", "doc_004", "doc_018", "doc_019", "doc_020", "doc_009", "doc_015", "doc_016", "doc_017"]
MUESTRA_CLIPS = ["clip_001", "clip_010", "clip_011", "clip_005", "clip_006", "clip_007", "clip_008", "clip_013", "clip_016", "clip_017"]

m_docs = docs.set_index("id").loc[MUESTRA_DOCS, ["tipo"]].assign(caracteres_ocr=docs.set_index("id")["texto_ocr"].str.len())
m_clips = clips.set_index("id").loc[MUESTRA_CLIPS, ["duracion_seg", "n_hablantes"]].join(fuentes[["tipo", "autor"]])
print("Documentos por tipo:", m_docs["tipo"].value_counts().to_dict())
print(f"Audios: {len(m_clips)} clips, {m_clips['duracion_seg'].sum() / 60:.1f} min, {m_clips['autor'].nunique()} fuentes, condiciones:", sorted(m_clips["tipo"].unique()))
display(m_docs, m_clips)

Documentos por tipo: {'manuscrito': 4, 'impreso': 3, 'foto': 3}
Audios: 10 clips, 11.5 min, 7 fuentes, condiciones: ['calle', 'fiesta, ruido', 'lectura, estudio', 'marcha, ruido', 'testimonio']


,tipo,caracteres_ocr
id,,
doc_001,impreso,362
doc_002,impreso,1038
doc_004,impreso,1516
doc_018,foto,2526
doc_019,foto,2388
doc_020,foto,504
doc_009,manuscrito,1885
doc_015,manuscrito,1872
doc_016,manuscrito,1130


,duracion_seg,n_hablantes,tipo,autor
id,,,,
clip_001,136.3,2,calle,JorgeFlores-pe
clip_010,96.1,1,calle,Gianpier Yair Venegas Ambia
clip_011,77.4,2,calle,Gianpier Yair Venegas Ambia
clip_005,81.6,2,"fiesta, ruido",Rashelley Perez
clip_006,77.4,2,"marcha, ruido",Hellloff233
clip_007,95.0,5,"marcha, ruido",Hellloff233
clip_008,54.3,3,"marcha, ruido",Joaquin Canales
clip_013,53.1,1,testimonio,Chantell Acosta
clip_016,10.2,1,"lectura, estudio","Google (Guevara-Rukoz et al., 2020)"


## 2. Referencias manuales

Las transcripciones manuales están en `datos/raw/referencias/<id>.txt`, que no se versiona. Las métricas se calculan con ese texto original. Para publicarlo en la columna `texto_referencia` de los CSV se aplica la misma seudonimización revisada de la Parte 1: los audios y los manuscritos se seudonimizan; los impresos y las fotos de periódicos públicos, no.

**Control de calidad.** Los dos clips de OpenSLR tienen una transcripción humana publicada por el propio corpus, y las referencias manuales coinciden palabra por palabra con ella (con la normalización de la sección 1). En los demás audios no hay una transcripción externa con qué comparar.

**Refuerzo de la seudonimización (27/10/2026).** Al verificar los textos de esta parte aparecieron variantes nuevas de nombres. Venían de las referencias, escritas correctamente, y de las salidas de los otros modelos, que deforman los nombres de otra manera. Se agregaron 19 variantes a la lista revisada (`ner, parte 2`). En los manuscritos se reemplaza también cada nombre o apellido suelto que se parezca a uno de la lista. Esto corrige además **15 restos de apellidos** que habían quedado en el texto OCR de 6 manuscritos publicado en la Parte 1. La verificación final, con la lista y con el NER, está al final de la sección 4.

**Corrección de una referencia (28/10/2026).** Durante la auditoría del LLM (sección 5) se encontró un error en la transcripción manual de `doc_016`: el año de la fecha de cierre se había transcrito como "veintitres", pero en la imagen dice "ochentitres" (1983). Se corrigió la referencia mirando la imagen y se recalculó el CER. La corrección afecta por igual a los 3 OCR y no cambia ninguna configuración de modelo.

In [2]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import jiwer
import plotly.express as px

import metricas
import voz

REF = Path("datos/raw/referencias")
referencia = {p.stem: p.read_text(encoding="utf-8") for p in sorted(REF.glob("*.txt")) if p.read_text(encoding="utf-8").strip()}
pendientes = [i for i in MUESTRA_DOCS + MUESTRA_CLIPS if i not in referencia]
print(f"Referencias manuales listas: {len(referencia)} de 20 · pendientes: {pendientes or 'ninguna'}")

atrib = pd.read_csv("datos/raw/audio/atribucion_audio.csv").set_index("id")
for c in ["clip_016", "clip_017"]:
    print(f"{c} vs transcripción oficial de OpenSLR: WER {metricas.wer([atrib.loc[c, 'transcripcion_openslr']], [referencia[c]]):.3f}")

reemplazos = pd.read_csv("datos/raw/seudonimizacion/reemplazos.csv")
activos = reemplazos[reemplazos["aplicar"] == "si"]
# los nombres más largos primero: "Augusto Nuñez del Prado Vera" antes que "Prado" suelto
mapa = dict(sorted(zip(activos["nombre"], activos["seudonimo"]), key=lambda kv: -len(kv[0])))
print("Seudonimización:", reemplazos.groupby(["origen", "aplicar"]).size().to_dict())
seudonimizar = lambda t: voz.seudonimizar(t, mapa) if isinstance(t, str) and t else t

# Manuscritos: además del nombre completo, cada nombre o apellido suelto. Los OCR leen mal los nombres y
# cada modelo los deforma distinto, así que quedaban restos como "de Leiva" junto a un seudónimo.
# Se compara cada palabra con mayúscula contra los componentes de los nombres revisados (parecido >= 75),
# salvo palabras comunes del registro que se parecen a algún componente.
from rapidfuzz import fuzz, process
componentes = {}
for nombre, seud in mapa.items():
    if "manuscrito" in activos.set_index("nombre").loc[[nombre], "fuente"].iloc[0]:
        for w in re.findall(r"\w+", nombre):
            if len(w) >= 4 and re.fullmatch(r"[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+", w):
                componentes.setdefault(w, seud)
COMUNES = {"Arancel", "Notario"}
CARGOS = ["Registrador", "Notario", "Juez", "Escribano"]
# una palabra con mayúscula justo después de un seudónimo es parte del mismo nombre ("PERSONA_28 Umeres")
PEGADA = re.compile(r"(PERSONA_\d+)((?:\s+(?:de|del|y|i))?\s+)(?!PERSONA_)([A-ZÁÉÍÓÚÑ][\wáéíóúñ]{2,})")

def seudonimizar_manuscrito(t):
    if not isinstance(t, str) or not t:
        return t
    def cambiar(m):
        w = m.group()
        r = None if w in COMUNES else process.extractOne(w, list(componentes), scorer=fuzz.ratio, score_cutoff=75)
        return componentes[r[0]] if r else w
    def pegada(m):
        cargo = process.extractOne(m.group(3), CARGOS, scorer=fuzz.ratio, score_cutoff=70)
        return m.group(0) if cargo else m.group(1) + m.group(2) + m.group(1)
    t = re.sub(r"\b[A-ZÁÉÍÓÚÑ][\wáéíóúñ]{3,}\b", cambiar, seudonimizar(t))
    for _ in range(3):
        t = PEGADA.sub(pegada, t)
    return t

# texto_referencia publicado: seudonimizado en audios y manuscritos
clips["texto_referencia"] = clips["id"].map(lambda i: seudonimizar(referencia.get(i, "")))
privado = docs["tipo"] == "manuscrito"
docs["texto_referencia"] = [seudonimizar_manuscrito(referencia.get(i, "")) if p else referencia.get(i, "") for i, p in zip(docs["id"], privado)]
# refuerzo sobre el texto OCR de la Parte 1: se publicó con restos de apellidos (ver nota arriba)
docs.loc[privado, "texto_ocr"] = docs.loc[privado, "texto_ocr"].map(seudonimizar_manuscrito)
clips.to_csv("datos/transcripciones.csv", index=False)
docs.to_csv("datos/documentos.csv", index=False)
print("texto_referencia completado en", (clips["texto_referencia"] != "").sum(), "clips y", (docs["texto_referencia"] != "").sum(), "documentos")

Referencias manuales listas: 20 de 20 · pendientes: ninguna
clip_016 vs transcripción oficial de OpenSLR: WER 0.000
clip_017 vs transcripción oficial de OpenSLR: WER 0.000
Seudonimización: {('ner', 'no'): 12, ('ner', 'si'): 48, ('ner, parte 2', 'si'): 19, ('ner, ronda 2', 'si'): 3}
texto_referencia completado en 10 clips y 10 documentos


## 3. Audio: línea base y modelos alternativos (WER y latencia)

Los 3 modelos transcribieron los mismos 10 clips con la configuración de la sección 1 (`evaluar_asr.py`, un proceso por modelo, uno tras otro). El WER global suma los errores de los 10 clips y los divide por el total de palabras de las referencias, así que los clips largos pesan más. El factor de tiempo real (RTF) global es el tiempo total de proceso dividido por la duración total del audio.

In [3]:
MODELOS_ASR = {"faster-whisper-small": "Whisper small (línea base)", "faster-whisper": "Whisper large-v3-turbo",
               "qwen3-asr-0.6b": "Qwen3-ASR 0.6B"}
EVAL = Path("datos/raw/audio/eval")
filas = []
for clave, nombre in MODELOS_ASR.items():
    for c in MUESTRA_CLIPS:
        d = json.loads((EVAL / clave / f"{c}.json").read_text(encoding="utf-8"))
        o = jiwer.process_words(metricas.normalizar(referencia[c]), metricas.normalizar(d["texto"]) or "∅")
        filas.append({"modelo": nombre, "id": c, "condicion": fuentes.loc[c, "tipo"], "duracion_seg": d["duracion_seg"],
                      "segundos": d["segundos"], "palabras_ref": o.hits + o.substitutions + o.deletions,
                      "sustituidas": o.substitutions, "borradas": o.deletions, "insertadas": o.insertions, "wer": o.wer,
                      "texto": d["texto"], "segmentos": d["segmentos"]})
asr = pd.DataFrame(filas)
asr["errores"] = asr[["sustituidas", "borradas", "insertadas"]].sum(axis=1)

def resumir(g):
    return pd.Series({"WER": g["errores"].sum() / g["palabras_ref"].sum(), "RTF": g["segundos"].sum() / g["duracion_seg"].sum(),
                      "sustituidas": g["sustituidas"].sum(), "borradas": g["borradas"].sum(), "insertadas": g["insertadas"].sum()})

tabla_asr = asr.groupby("modelo", sort=False).apply(resumir, include_groups=False)
display(tabla_asr.round(3))
por_condicion = asr.groupby(["condicion", "modelo"], sort=False).apply(resumir, include_groups=False)["WER"].unstack()
display(por_condicion.round(3))
display(asr.pivot(index=["id", "condicion"], columns="modelo", values="wer").round(3))

,WER,RTF,sustituidas,borradas,insertadas
modelo,,,,,
Whisper small (línea base),0.320,0.531,174.0,65.0,197.0
Whisper large-v3-turbo,0.302,1.822,122.0,53.0,237.0
Qwen3-ASR 0.6B,0.379,0.949,206.0,63.0,248.0


modelo,Whisper small (línea base),Whisper large-v3-turbo,Qwen3-ASR 0.6B
condicion,,,
calle,0.264,0.233,0.343
"fiesta, ruido",0.441,0.419,0.520
"marcha, ruido",0.400,0.410,0.437
testimonio,0.183,0.139,0.209
"lectura, estudio",0.037,0.037,0.037


,modelo,Qwen3-ASR 0.6B,Whisper large-v3-turbo,Whisper small (línea base)
id,condicion,,,
clip_001,calle,0.387,0.281,0.293
clip_005,"fiesta, ruido",0.520,0.419,0.441
clip_006,"marcha, ruido",0.265,0.193,0.283
clip_007,"marcha, ruido",0.664,0.671,0.503
clip_008,"marcha, ruido",0.406,0.398,0.438
clip_010,calle,0.356,0.207,0.264
clip_011,calle,0.267,0.188,0.222
clip_013,testimonio,0.209,0.139,0.183
clip_016,"lectura, estudio",0.067,0.067,0.067


In [4]:
fig = px.bar(por_condicion.reset_index().melt(id_vars="condicion", var_name="modelo", value_name="WER"),
             x="condicion", y="WER", color="modelo", barmode="group", text_auto=".2f",
             title="El ruido de marchas y fiestas multiplica el error: el WER sube de 0.04 en estudio a 0.40–0.52",
             labels={"condicion": "Condición de grabación"})
fig.show()

fig = px.scatter(tabla_asr.reset_index(), x="RTF", y="WER", text="modelo", size=[1, 1, 1],
                 title="Whisper turbo apenas mejora a small (0.30 vs 0.32) pero tarda 3,4 veces más",
                 labels={"RTF": "Factor de tiempo real (segundos de proceso / segundo de audio)"})
fig.update_traces(textposition="top center")
fig.show()

### Alucinaciones en silencio y referencias incompletas

En los 3 modelos predominan las palabras **insertadas**. Puede haber dos causas:

1. **Alucinaciones.** En los últimos ~20 s de `clip_007`, que son casi silencio, Whisper turbo escribe la misma palabra decenas de veces, aunque el VAD estaba activado.
2. **Referencias incompletas.** En varios tramos los 3 modelos coinciden en palabras que no están en la referencia manual, sobre todo preguntas del entrevistador, voces de fondo y repeticiones. Se marcaron esos tramos para volver a escucharlos, pero se decidió no modificar las referencias.

Esta segunda causa es una **limitación**: si a la referencia le faltan palabras dichas, las tres métricas suben por igual. Por eso el WER absoluto puede estar sobreestimado, aunque la comparación entre modelos sigue siendo válida porque todos se miden contra el mismo texto.

In [5]:
def repeticiones(segmentos, minimo=3):
    # segmentos idénticos seguidos (la firma típica de una alucinación de Whisper)
    textos = [metricas.normalizar(s[2]) for s in segmentos]
    rachas, n = [], 1
    for a, b in zip(textos, textos[1:] + [None]):
        if a == b:
            n += 1
        else:
            if n >= minimo:
                rachas.append((a, n))
            n = 1
    return rachas

for (modelo, c), segs in asr.set_index(["modelo", "id"])["segmentos"].items():
    for texto, n in repeticiones(segs):
        print(f"{modelo} · {c}: '{texto}' repetido {n} veces seguidas")

turbo7 = asr[(asr["modelo"] == "Whisper large-v3-turbo") & (asr["id"] == "clip_007")].iloc[0]
sin_cola = " ".join(t for ini, fin, t in turbo7["segmentos"] if ini < 93.5)
print(f"Whisper turbo en clip_007: WER {turbo7['wer']:.3f} → {metricas.wer([referencia['clip_007']], [sin_cola]):.3f} sin la cola alucinada")
print("Palabras insertadas / errores totales:", (asr.groupby("modelo", sort=False)["insertadas"].sum() / asr.groupby("modelo", sort=False)["errores"].sum()).round(2).to_dict())

Whisper large-v3-turbo · clip_007: 'listo' repetido 23 veces seguidas
Whisper turbo en clip_007: WER 0.671 → 0.510 sin la cola alucinada
Palabras insertadas / errores totales: {'Whisper small (línea base)': 0.45, 'Whisper large-v3-turbo': 0.58, 'Qwen3-ASR 0.6B': 0.48}


In [6]:
# resultados por clip para el dashboard: transcripciones seudonimizadas, sin nombres reales
pub = asr.assign(texto=asr["texto"].map(seudonimizar), rtf=asr["segundos"] / asr["duracion_seg"])
pub[["id", "modelo", "condicion", "duracion_seg", "segundos", "rtf", "wer", "sustituidas", "borradas", "insertadas", "texto"]].to_csv(
    "datos/evaluacion_asr.csv", index=False)
print("datos/evaluacion_asr.csv:", pub.shape)

# verificación: ningún nombre revisado con aplicar=si queda en lo publicado
publicado = " ".join(pd.read_csv(p).astype(str).agg(" ".join, axis=1).str.cat(sep=" ")
                     for p in ["datos/evaluacion_asr.csv", "datos/transcripciones.csv"])
publicado += " " + " ".join(docs.loc[privado, "texto_referencia"].fillna(""))
quedan = [n for n in mapa if re.search(rf"(?<!\w){re.escape(n)}(?!\w)", publicado)]
print("Nombres reales en textos publicados de audio y manuscritos:", len(quedan))

datos/evaluacion_asr.csv: (30, 14)
Nombres reales en textos publicados de audio y manuscritos: 0


## 4. Documentos: línea base y modelos alternativos (CER)

Los 3 OCR leyeron los mismos 10 recortes (`ocr_corpus.py --recortes`). El CER global suma los errores de caracteres de los 10 recortes y los divide por el total de caracteres de las referencias. El CER **es sensible al orden**: si un OCR lee bien las palabras pero mezcla las columnas, el CER sube. Por eso se agrega una métrica complementaria, que **no sirve para elegir modelo**: el porcentaje de palabras de la referencia que aparecen en la salida sin importar el orden. Es lo que importa para un archivo **buscable**.

En `datos/documentos.csv`, la columna `texto_referencia` contiene la transcripción manual **del recorte evaluado**, no de la página completa.

In [7]:
MODELOS_OCR = {"paddle-base": "PaddleOCR por defecto (línea base)", "paddle": "PaddleOCR ajustado (Parte 1)",
               "tesseract": "Tesseract 5.5"}
OCR = Path("datos/raw/documentos/ocr")

def palabras_encontradas(ref, hip):
    r, h = [pd.Series(metricas.normalizar(t).split()).value_counts() for t in (ref, hip)]
    return r.combine(h.reindex(r.index, fill_value=0), min).sum() / r.sum()

filas = []
for clave, nombre in MODELOS_OCR.items():
    for d in MUESTRA_DOCS:
        j = json.loads((OCR / f"{clave}-recorte" / f"{d}.json").read_text(encoding="utf-8"))
        hip = "\n".join(l["texto"] for l in j["lineas"])
        o = jiwer.process_characters(metricas.normalizar(referencia[d]), metricas.normalizar(hip) or "∅")
        filas.append({"modelo": nombre, "id": d, "tipo": docs.set_index("id").loc[d, "tipo"], "segundos": j["segundos"],
                      "caracteres_ref": o.hits + o.substitutions + o.deletions,
                      "errores": o.substitutions + o.deletions + o.insertions, "cer": o.cer,
                      "palabras_encontradas": palabras_encontradas(referencia[d], hip), "texto": hip})
ocr = pd.DataFrame(filas)

def resumir_ocr(g):
    return pd.Series({"CER": g["errores"].sum() / g["caracteres_ref"].sum(), "palabras_encontradas": g["palabras_encontradas"].mean(),
                      "segundos_por_doc": g["segundos"].mean()})

tabla_ocr = ocr.groupby("modelo", sort=False).apply(resumir_ocr, include_groups=False)
display(tabla_ocr.round(3))
por_tipo = ocr.groupby(["tipo", "modelo"], sort=False).apply(resumir_ocr, include_groups=False)["CER"].unstack()
display(por_tipo.round(3))
display(ocr.pivot(index=["id", "tipo"], columns="modelo", values="cer").round(3))

,CER,palabras_encontradas,segundos_por_doc
modelo,,,
PaddleOCR por defecto (línea base),0.417,0.546,65.19
PaddleOCR ajustado (Parte 1),0.352,0.590,32.31
Tesseract 5.5,0.379,0.581,1.53


modelo,PaddleOCR por defecto (línea base),PaddleOCR ajustado (Parte 1),Tesseract 5.5
tipo,,,
impreso,0.073,0.009,0.009
foto,0.482,0.437,0.236
manuscrito,0.565,0.487,0.674


,modelo,PaddleOCR ajustado (Parte 1),PaddleOCR por defecto (línea base),Tesseract 5.5
id,tipo,,,
doc_001,impreso,0.000,0.000,0.000
doc_002,impreso,0.002,0.128,0.002
doc_004,impreso,0.024,0.072,0.024
doc_009,manuscrito,0.380,0.468,0.653
doc_015,manuscrito,0.475,0.579,0.669
doc_016,manuscrito,0.551,0.635,0.623
doc_017,manuscrito,0.552,0.592,0.737
doc_018,foto,0.658,0.639,0.146
doc_019,foto,0.375,0.423,0.510


In [8]:
fig = px.bar(por_tipo.reset_index().melt(id_vars="tipo", var_name="modelo", value_name="CER"),
             x="tipo", y="CER", color="modelo", barmode="group", text_auto=".2f",
             title="Ningún OCR gana en todo: Tesseract lidera en fotos, PaddleOCR en manuscritos y en impresos casi no hay errores",
             labels={"tipo": "Tipo de documento"})
fig.show()

### El orden de lectura en las fotos de periódico

En `doc_018` PaddleOCR alcanza un CER de ~0.65 y Tesseract de ~0.15. La diferencia no está en reconocer letras sueltas. Tiene dos causas:

1. **Títulos decorativos.** PaddleOCR no lee los títulos grandes de la cabecera, en tipografía decorativa del siglo XIX.
2. **Orden de lectura.** Intercala líneas de la columna derecha antes de terminar la izquierda.

El análisis de página de Tesseract (PSM 3) separa mejor las columnas. Abajo se muestran las primeras líneas en el orden en que salen de cada modelo. Este documento es una publicación histórica pública, así que no tiene datos personales.

In [9]:
for nombre in ["PaddleOCR ajustado (Parte 1)", "Tesseract 5.5"]:
    fila = ocr[(ocr["modelo"] == nombre) & (ocr["id"] == "doc_018")].iloc[0]
    print(f"{nombre} · CER {fila['cer']:.2f} · palabras encontradas {fila['palabras_encontradas']:.0%}")
    print("   " + " | ".join(l[:28] for l in fila["texto"].splitlines()[:12]))
print("Referencia:\n   " + " | ".join(l[:28] for l in referencia["doc_018"].splitlines()[:10]))

PaddleOCR ajustado (Parte 1) · CER 0.66 · palabras encontradas 66%
   OICLAE |  |  | memartaemto |  | CUZCO, T DE ENERO DE IS57. | [N.1. | Parte oficial. | sino su fugn ou direcoion a  | te y que los lenles Torateos | habian perseguido hasta tres | de distancia tirotenndolos, 
Tesseract 5.5 · CER 0.15 · palabras encontradas 81%
   y 7 | f | f | REGISTRO OFICIAL | Z DI L | Departamento, | Numero estraordinario. | pasee ——]]]]—]——— | CUZCO, 7 DE ENERO DE 1857. | eee aaa CEE CE as | Parte oficial. | Republica Perunna— Estado Ma
Referencia:
   Tomo 9º  | Registro oficial | del | departamento. | Numero estraordinario. | Cuzco, 7 de enero de 1857.[N | Parte oficial. | Republica Peruana-Estado May | sino su fuga en direccion a  | Me cabe pues, ha satisfaccio


In [10]:
# resultados por documento para el dashboard: manuscritos seudonimizados
pub = ocr.assign(texto=[seudonimizar_manuscrito(t) if tp == "manuscrito" else t for t, tp in zip(ocr["texto"], ocr["tipo"])])
pub[["id", "modelo", "tipo", "segundos", "cer", "palabras_encontradas", "texto"]].to_csv("datos/evaluacion_ocr.csv", index=False)
print("datos/evaluacion_ocr.csv:", pub.shape)

# verificación de privacidad de todo lo que publica la Parte 2 en audio y manuscritos:
# referencias manuales y salidas de los 6 modelos, ya seudonimizadas
textos_pub = dict(zip(docs.loc[privado, "id"], docs.loc[privado, "texto_referencia"].fillna("")))
textos_pub |= dict(zip("ref·" + clips["id"], clips["texto_referencia"].fillna("")))
textos_pub |= {f"{r.id}·{r.modelo}": r.texto for r in pub[pub["tipo"] == "manuscrito"].itertuples()}
textos_pub |= {f"{r.id}·{r.modelo}": r.texto for r in pd.read_csv("datos/evaluacion_asr.csv").fillna("").itertuples()}
publicado = " ".join(textos_pub.values())
print("Nombres de la lista revisada que siguen apareciendo:", sum(bool(re.search(rf"(?<!\w){re.escape(n)}(?!\w)", publicado)) for n in mapa))
with tempfile.TemporaryDirectory() as tmp:
    entrada, salida = Path(tmp) / "in.json", Path(tmp) / "out.json"
    entrada.write_text(json.dumps(textos_pub, ensure_ascii=False), encoding="utf-8")
    subprocess.run([sys.executable, "ner_personas.py", entrada, salida], check=True, capture_output=True)
    quedan = {k: [n for n in v if "PERSONA" not in n] for k, v in json.loads(salida.read_text(encoding="utf-8")).items()}
print("Personas que el NER aún detecta en los textos publicados:", sum(len(v) for v in quedan.values()))
for k, v in quedan.items():
    if v:
        print(f"  {k}: {len(v)}")

datos/evaluacion_ocr.csv: (30, 9)
Nombres de la lista revisada que siguen apareciendo: 0
Personas que el NER aún detecta en los textos publicados: 1
  clip_008·Whisper large-v3-turbo: 1


## 5. Extracción estructurada con LLM y auditoría humana

`extraer_llm.py` envía a Gemini los 45 textos **ya seudonimizados**: los `texto_ocr` de `documentos.csv` y los `texto_asr` de `transcripciones.csv`. Gemini devuelve un JSON con esquema fijo, temperatura 0 y `null` para los datos ausentes.

Hubo dos cambios respecto del plan de la sección 1, y ninguno se debió a los resultados:
- **Modelo:** se cambió `gemini-3.8-flash` por **`gemini-3.5-flash`** y se enviaron **3 textos por solicitud**. La cuota gratuita es de 20 solicitudes diarias por modelo, y la de 3.8 se agotó a los 13 documentos; esas respuestas se descartaron para no mezclar configuraciones.
- **Manuscritos:** se extrajeron de nuevo desde el texto corregido por el refuerzo de seudonimización (sección 2), para que ningún resto de apellido llegara al resultado.

El resultado está en `datos/extraccion_llm.csv`, con las columnas `id`, `modalidad`, `campos_json`, `modelo_llm` y `fecha`.

In [11]:
ext = pd.read_csv("datos/extraccion_llm.csv")
campos = pd.DataFrame([json.loads(c) | {"id": i, "mod": m} for i, m, c in ext[["id", "modalidad", "campos_json"]].itertuples(index=False)])
print("Extracciones:", ext["modalidad"].value_counts().to_dict(), "· modelo:", ext["modelo_llm"].unique().tolist())
nulos = campos.groupby("mod").agg(lambda s: s.isna().sum()).drop(columns="id")
print("Campos en null por modalidad (los que no corresponden a la modalidad quedan en null por diseño):")
display(nulos)
# ejemplos de documentos públicos (impreso y foto) y de un audio
display(campos.set_index("id").loc[["doc_001", "doc_018", "clip_016"]].drop(columns="mod").T)

Extracciones: {'documento': 25, 'audio': 20} · modelo: ['gemini-3.5-flash']
Campos en null por modalidad (los que no corresponden a la modalidad quedan en null por diseño):


,modalidad,tema,resumen,numero_hablantes,citas_clave,emisor,fecha,lugar,monto
mod,,,,,,,,,
audio,0,0,0,0,0,20,20,20,20
documento,0,0,0,25,25,5,9,13,15


id,doc_001,doc_018,clip_016
modalidad,documento,documento,audio
tema,Catálogo de periódicos cusqueños,Parte oficial de acciones militares,Origen del chicle en México
resumen,Presentación de la portada del catálogo de per...,Informe oficial remitido al Gran Mariscal sobr...,Se menciona de forma breve la llegada y conoci...
numero_hablantes,NaN,NaN,1.0
citas_clave,NaN,NaN,[El chicle se conoció en México durante la inv...
emisor,Universidad Nacional San Antonio Abad del Cusco,Nicolas Freire,NaN
fecha,None,7 DE ENERO DE LS57,NaN
lugar,Cusco,CUZCO,NaN
monto,None,None,NaN


### Auditoría humana de 20 extracciones

`preparar_auditoria.py` sorteó, con la semilla 2026, 10 documentos de 25 y 10 audios de 20. Cada campo se marcó a mano en `datos/auditoria_llm.csv` con `si` (correcto), `no` (incorrecto) o `na` (no corresponde a la modalidad). El criterio se juzga **contra el texto que recibió el LLM**, porque es lo único que conocía:

- **`tema` y `resumen`:** describen el texto sin agregar hechos que no estén en él.
- **`emisor`, `fecha`, `lugar` y `monto`:** son correctos si el dato del **documento** está en el texto, aunque traiga los errores de lectura del OCR, o si quedan en `null` porque el dato no aparece. Inventar un dato, tomar uno secundario o dejar `null` un dato que sí está cuenta como error.
- **`numero_hablantes`:** debe coincidir con las marcas de hablante distintas del texto.
- **`citas_clave`:** deben estar copiadas del texto y ser representativas.

Una extracción es **completamente correcta** si todos sus campos están en `si` o `na`.

In [12]:
aud = pd.read_csv("datos/auditoria_llm.csv").fillna("")
CAMPOS_AUD = ["tema", "resumen", "emisor", "fecha", "lugar", "monto", "numero_hablantes", "citas_clave"]
aud["correcta"] = aud[CAMPOS_AUD].apply(lambda f: all(v in ("si", "na") for v in f), axis=1)
acuerdo = int(aud["correcta"].sum())
print(f"Acuerdo LLM-Humano: {acuerdo}/20 ({acuerdo / 20:.0%}) extracciones completamente correctas")
print("Por modalidad:", aud.groupby("modalidad")["correcta"].agg(lambda s: f"{s.sum()}/{len(s)}").to_dict())
por_campo = pd.DataFrame({c: {"correctos": (aud[c] == "si").sum(), "evaluados": (aud[c] != "na").sum()} for c in CAMPOS_AUD}).T
por_campo["acierto"] = (por_campo["correctos"] / por_campo["evaluados"]).round(2)
display(por_campo)
display(aud.loc[~aud["correcta"] | (aud["comentario"] != ""), ["id"] + CAMPOS_AUD + ["comentario"]])

Acuerdo LLM-Humano: 17/20 (85%) extracciones completamente correctas
Por modalidad: {'audio': '10/10', 'documento': '7/10'}


,correctos,evaluados,acierto
tema,20,20,1.0
resumen,20,20,1.0
emisor,9,10,0.9
fecha,8,10,0.8
lugar,9,10,0.9
monto,9,10,0.9
numero_hablantes,10,10,1.0
citas_clave,10,10,1.0


,id,tema,resumen,emisor,fecha,lugar,monto,numero_hablantes,citas_clave,comentario
5,doc_016,si,si,si,no,si,si,na,na,
6,doc_017,si,si,si,no,no,si,na,na,tomó la fecha de la anotación marginal (contin...
7,doc_018,si,si,si,si,si,si,na,na,copió el error del OCR (LS57 por 1857)
9,doc_025,si,si,no,si,si,no,na,na,página con varios decretos: emisor y monto cor...
12,clip_010,si,si,na,na,na,na,si,si,en el audio hay 2 voces; la diarización solo d...


### Ejemplos de errores y errores típicos del LLM

Los 3 casos con errores son documentos:

| Documento | Campo | Qué pasó |
|---|---|---|
| `doc_016` (manuscrito) | `fecha` | El OCR deformó la fecha de cierre ("Gureooieinuve emro d md Ccreientis clonto ties"). Gemini la **descifró** como "diecinueve de enero de mil novecientos", pero **cortó el año**: falta "ochentitres" (1983), que sí deja rastro en el texto. |
| `doc_017` (manuscrito) | `fecha`, `lugar` | Tomó la fecha de una **anotación marginal** (de dónde continúa la partida) en vez de la del asiento, y dejó `lugar` en `null` aunque el asiento lo menciona. |
| `doc_025` (foto) | `emisor`, `monto` | La página reúne **varios decretos** de distintas autoridades. Gemini eligió la firma de uno solo como emisor de toda la página, y un monto que reconstruyó juntando fragmentos de columnas que el OCR había mezclado. |

Otros casos marcados como correctos muestran el mismo comportamiento:
- **Inconsistencia con los errores del OCR.** En `doc_018` Gemini **copió** el error "LS57" (por 1857), como se le pidió. En `doc_019` lo **corrigió**: el OCR decía "T857" y Gemini puso "1857". No se comporta igual en casos parecidos.
- **Documentos que contienen otros documentos.** En `doc_018` la fecha y el lugar son los de la edición del periódico, pero el parte oficial reproducido es de Alto de la Villa, 30 de diciembre de 1856. En `doc_009` hay varios asientos con distintos montos y Gemini tomó el de la compraventa.

**Conclusión.** El LLM **tolera muy bien el ruido del OCR** para captar el `tema` y el `resumen`: lo hizo incluso en manuscritos con CER cercano a 0.5, porque reconoce palabras deformadas y el "molde" de los documentos registrales. Esa misma capacidad de completar es la que produce los errores en los campos puntuales:
1. **descifra o recorta** datos deformados;
2. **no distingue** el texto principal de las anotaciones;
3. **fuerza una sola ficha** en páginas que reúnen varios documentos.

Para un archivo buscable convendría:
- una ficha por documento, y no por página;
- separar *fecha de publicación* de *fecha del hecho*;
- marcar en la ficha si un dato fue copiado o interpretado.

**Hallazgo sobre la diarización (no es un error del LLM).** En `clip_010` Gemini contó bien las marcas de hablante (1), pero quien audita escuchó **2 voces**: la diarización de la Parte 1 no separó al entrevistador. `n_hablantes` subestima el número real de voces en entrevistas con preguntas breves.

## 6. Casos difíciles

### Manuscritos
Son el peor caso de los tres OCR:
- el CER de los manuscritos va de **0.38 a 0.74**, según el documento y el modelo, frente a **0.01** en los impresos;
- los modelos están entrenados sobre todo con texto impreso, y la letra cursiva de los asientos registrales (1911–1995), con trazos ligados y abreviaturas, queda fuera de lo que aprendieron;
- PaddleOCR ajustado es el que mejor resiste. Tesseract, con su segmentación de página, a veces pierde líneas enteras (`doc_017`: 0.74).

### Audio con ruido o música
- En **marchas y fiestas** el WER es de **0.40 a 0.52**, frente a **0.04** en estudio y **0.14 a 0.34** en calle y testimonio.
- La causa son las voces superpuestas, las arengas y la música de fondo.
- Whisper turbo además **alucinó** "listo" 23 veces en el silencio final de `clip_007`, aunque el VAD estaba activado.
- Qwen3-ASR es el más sensible al ruido (0.52 en la fiesta).

### Quechua de Huaylas con MMS (bonificación)
No se realizó.

In [13]:
dificil_ocr = ocr[ocr["tipo"] == "manuscrito"].pivot(index="id", columns="modelo", values="cer").round(3)
dificil_ocr.loc["CER global manuscritos"] = por_tipo.loc["manuscrito"].round(3)
display(dificil_ocr)
dificil_asr = asr[asr["condicion"].isin(["marcha, ruido", "fiesta, ruido"])].pivot(index=["id", "condicion"], columns="modelo", values="wer").round(3)
display(dificil_asr)

modelo,PaddleOCR ajustado (Parte 1),PaddleOCR por defecto (línea base),Tesseract 5.5
id,,,
doc_009,0.380,0.468,0.653
doc_015,0.475,0.579,0.669
doc_016,0.551,0.635,0.623
doc_017,0.552,0.592,0.737
CER global manuscritos,0.487,0.565,0.674


,modelo,Qwen3-ASR 0.6B,Whisper large-v3-turbo,Whisper small (línea base)
id,condicion,,,
clip_005,"fiesta, ruido",0.520,0.419,0.441
clip_006,"marcha, ruido",0.265,0.193,0.283
clip_007,"marcha, ruido",0.664,0.671,0.503
clip_008,"marcha, ruido",0.406,0.398,0.438


## 7. Comparativa final y decisión

La tabla reúne las métricas de todos los modelos sobre **las mismas 10 muestras de cada modalidad**, con su costo de cómputo. Todo se ejecutó en CPU, en un Intel i5 de 2 núcleos.

In [14]:
comparativa = pd.concat([
    tabla_ocr.reset_index().rename(columns={"CER": "error", "segundos_por_doc": "latencia"}).assign(
        modalidad="documentos", metrica="CER", unidad_latencia="s por documento"),
    tabla_asr.reset_index().rename(columns={"WER": "error", "RTF": "latencia"}).assign(
        modalidad="audio", metrica="WER", unidad_latencia="factor de tiempo real"),
])[["modalidad", "modelo", "metrica", "error", "latencia", "unidad_latencia"]]
comparativa["rol"] = comparativa["modelo"].map(lambda m: "línea base" if "base" in m else "alternativo")
display(comparativa.round(3))

# esquema elegido para documentos: Tesseract en impresos y fotos, PaddleOCR ajustado en manuscritos
elegido = ocr[((ocr["modelo"] == "Tesseract 5.5") & (ocr["tipo"] != "manuscrito"))
              | ((ocr["modelo"] == "PaddleOCR ajustado (Parte 1)") & (ocr["tipo"] == "manuscrito"))]
cer_elegido = elegido["errores"].sum() / elegido["caracteres_ref"].sum()
wer_elegido = tabla_asr.loc["Whisper small (línea base)", "WER"]
print(f"Documentos, esquema por tipo (Tesseract + PaddleOCR en manuscritos): CER {cer_elegido:.3f}")
print("  (optimista: el reparto por tipo se eligió mirando esta misma muestra)")
print(f"Documentos, un solo modelo (Tesseract): CER {tabla_ocr.loc['Tesseract 5.5', 'CER']:.3f}")
print(f"Audio, Whisper small: WER {wer_elegido:.3f}")
print(f"\nResultados Test: WER: {wer_elegido:.2f} | CER: {tabla_ocr.loc['Tesseract 5.5', 'CER']:.2f} | Acuerdo LLM-Humano: {acuerdo}/20")

# métricas resumidas para el dashboard (Parte 3)
grupos = pd.concat([por_tipo.reset_index().melt(id_vars="tipo", var_name="modelo", value_name="error").rename(columns={"tipo": "grupo"}).assign(metrica="CER"),
                    por_condicion.reset_index().melt(id_vars="condicion", var_name="modelo", value_name="error").rename(columns={"condicion": "grupo"}).assign(metrica="WER")])
comparativa.round(4).to_csv("datos/metricas_modelos.csv", index=False)
grupos.round(4).to_csv("datos/metricas_por_grupo.csv", index=False)
pd.DataFrame([{"acuerdo_llm_humano": acuerdo, "auditadas": len(aud)}]).to_csv("datos/metricas_llm.csv", index=False)
print("guardado: datos/metricas_modelos.csv, datos/metricas_por_grupo.csv, datos/metricas_llm.csv")

,modalidad,modelo,metrica,error,latencia,unidad_latencia,rol
0,documentos,PaddleOCR por defecto (línea base),CER,0.417,65.190,s por documento,línea base
1,documentos,PaddleOCR ajustado (Parte 1),CER,0.352,32.310,s por documento,alternativo
2,documentos,Tesseract 5.5,CER,0.379,1.530,s por documento,alternativo
0,audio,Whisper small (línea base),WER,0.320,0.531,factor de tiempo real,línea base
1,audio,Whisper large-v3-turbo,WER,0.302,1.822,factor de tiempo real,alternativo
2,audio,Qwen3-ASR 0.6B,WER,0.379,0.949,factor de tiempo real,alternativo


Documentos, esquema por tipo (Tesseract + PaddleOCR en manuscritos): CER 0.294
  (optimista: el reparto por tipo se eligió mirando esta misma muestra)
Documentos, un solo modelo (Tesseract): CER 0.379
Audio, Whisper small: WER 0.320

Resultados Test: WER: 0.32 | CER: 0.38 | Acuerdo LLM-Humano: 17/20
guardado: datos/metricas_modelos.csv, datos/metricas_por_grupo.csv, datos/metricas_llm.csv


### Decisión para un entorno real

**Audio: Whisper `small`.**
- Comete casi los mismos errores que `large-v3-turbo` (WER 0.32 frente a 0.30) y es **3,4 veces más rápido** (RTF 0.53 frente a 1.82). En CPU transcribe más rápido que el tiempo real; turbo tarda casi el doble de lo que dura el audio.
- La diferencia de precisión se concentra en la calle y los testimonios, y no compensa el costo. Turbo solo convendría con GPU, donde la latencia deja de ser un problema, y aun así hay que vigilar sus alucinaciones en silencios.
- Qwen3-ASR queda descartado: es más lento que small y menos preciso en el conjunto (0.38 frente a 0.32) y en cada condición salvo en estudio, donde empatan. Solo gana en 2 clips de marcha (`clip_006` y `clip_008`).

**Documentos: Tesseract como modelo único.**
- Su CER es 0.379, frente a 0.352 de PaddleOCR ajustado, y es **unas 20 veces más rápido** (1,5 s frente a 32 s por documento).
- Lee mejor las fotos de periódico, porque separa las columnas, y empata en los impresos.
- Su punto débil son los manuscritos (0.67 frente a 0.49). Si el archivo tiene muchos, conviene un **esquema por tipo**, con PaddleOCR ajustado solo para los manuscritos.
- La línea base de PaddleOCR, con el preprocesado por defecto, es la peor y la más lenta. El desdoblado y la orientación automáticos no ayudaron con estas imágenes.
- En los manuscritos ningún OCR es usable sin revisión humana.

**LLM.** Gemini sirve para indexar por `tema` y `resumen`: los 20 temas y resúmenes auditados fueron correctos. Los campos puntuales (`fecha`, `emisor`, `monto`, `lugar`) necesitan revisión en manuscritos y en páginas con varios documentos.

### Reflexión por tipo
| Tipo | Mejor desempeño | Peor desempeño | Causa |
|---|---|---|---|
| Impresos | CER 0.00–0.02 (Tesseract y PaddleOCR ajustado) | 0.13 (PaddleOCR base en `doc_002`) | Tipografía moderna y nítida. El preprocesado por defecto de Paddle reordena o pierde líneas en páginas con mucho espacio en blanco. |
| Fotos de periódico (1857–1862) | 0.13–0.15 (Tesseract) | 0.66 (PaddleOCR en `doc_018`) | Sombra, inclinación, tipografía decorativa en las cabeceras y dos columnas. Paddle mezcla las columnas. |
| Manuscritos | 0.38 (PaddleOCR ajustado, `doc_009`) | 0.74 (Tesseract, `doc_017`) | Letra cursiva y modelos entrenados con texto impreso. |
| Audio de estudio | WER 0.00–0.07 | — | Lectura clara y sin ruido. |
| Calle y testimonio | 0.14 (turbo en `clip_013`) | 0.39 (Qwen en `clip_001`) | Habla espontánea y ruido de ambiente. |
| Marchas y fiestas | 0.19 (turbo en `clip_006`) | 0.67 (turbo en `clip_007`) | Voces superpuestas, música y arengas. En `clip_007`, alucinación en el silencio final. |